In [ ]:
import json
import os
import time
from datetime import datetime

from dotenv import load_dotenv
from google.api_core.exceptions import DeadlineExceeded
from google.cloud import storage
from google.oauth2 import service_account

# Load environment variables from .env file
load_dotenv()
# Initialize ScriptMonitor if needed
# from core.script_monitor import ScriptMonitor
# monitor = ScriptMonitor('Crypto Fundamentals', database_name='officefield_local')

# Load credentials from environment variable
GCP_FILE_CRED = os.getenv("GCP_FILE_CRED")
GCP_FOLDER_PATH = "news_clustering"

credentials_dict = json.loads(GCP_FILE_CRED)
credentials = service_account.Credentials.from_service_account_info(credentials_dict)
client = storage.Client(project="stock_data", credentials=credentials)
bucket = client.bucket("eod_stock_data")

In [ ]:

main_path = os.getcwd()
FILE_NAMES = [
    "cluster_centroids.pkl",
    "cluster_info.pkl",
    "last_processed_date.txt",
]  # Replace with actual file names


In [ ]:
# Upload .pkl files to GCP
def upload_pkl_files():
    tried = True
    try:
        local_file_paths = [
            main_path + "/output_folder/cluster_centroids.pkl",
            main_path + "/output_folder/cluster_info.pkl",
            main_path + "/output_folder/last_processed_date.txt",
        ]
        for file_path in local_file_paths:
            print(file_path)
            file_name = os.path.basename(file_path)
            blob = bucket.blob(f"{GCP_FOLDER_PATH}/{file_name}")
            # blob.upload_from_filename(file_path)
            blob.upload_from_filename(file_path, timeout=600)
            print(f"Uploaded {file_path} to {GCP_FOLDER_PATH}/{file_name}")

    except Exception as e:
        print(f"Timeout exceeded while uploading {file_name}. Retrying...")
        time.sleep(5)  # Wait for 5 seconds before retrying
        blob.upload_from_filename(file_path, timeout=600)


In [ ]:

# Download .pkl files from GCP
def download_pkl_files():
    destination_dir = main_path + "/output_folder"
    os.makedirs(
        destination_dir, exist_ok=True
    )  # Create local destination directory if it doesn’t exist

    try:
        for file_name in FILE_NAMES:
            blob = bucket.blob(f"{GCP_FOLDER_PATH}/{file_name}")
            destination_path = os.path.join(destination_dir, file_name)

            # Check if blob exists and download it
            if blob.exists():
                blob.download_to_filename(destination_path)
                print(f"Downloaded {GCP_FOLDER_PATH}/{file_name} to {destination_path}")
            else:
                print(f"{GCP_FOLDER_PATH}/{file_name} does not exist in the bucket.")

    except Exception as e:
        print(f"Error downloading files: {e}")